# 04. Train / Valid / Test Split

**입력**: `data/processed/{class}/` — 사용자가 정제한 이미지 (원본 절대 수정 안 함)

**출력**: `data/split/{train,valid,test}/{class}/` — 학습용 복사본

**비율**: 8 : 1 : 1  |  **Seed**: 42  |  **Stratified split**

**실행 순서**: 셀을 위에서 아래로 `Shift+Enter`로 순서대로 실행하세요.

In [1]:
import os
import sys
import shutil
from pathlib import Path
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

SPLIT_DIR = config.SPLIT_DIR
SEED      = config.SPLIT_SEED
print(f'PROCESSED_DIR : {config.PROCESSED_DIR}')
print(f'SPLIT_DIR     : {SPLIT_DIR}')
print(f'SEED          : {SEED}')

PROCESSED_DIR : data\processed
SPLIT_DIR     : data\split
SEED          : 42


In [2]:
# ── PROJECT_LABELS vs 실제 processed 폴더 일치 검증 ────────────────────────────
# PROJECT_LABELS와 data/processed/ 폴더가 어긋났을 때 즉시 에러로 알려줌
from config import PROJECT_LABELS

actual_labels = sorted([
    d for d in os.listdir(config.PROCESSED_DIR)
    if os.path.isdir(os.path.join(config.PROCESSED_DIR, d))
    and len([
        f for f in os.listdir(os.path.join(config.PROCESSED_DIR, d))
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]) > 0
])
expected_labels = sorted(PROJECT_LABELS)

assert actual_labels == expected_labels, (
    f'PROJECT_LABELS와 실제 processed 폴더가 다릅니다.\n'
    f'PROJECT_LABELS : {expected_labels}\n'
    f'processed 폴더 : {actual_labels}'
)
print(f'PROJECT_LABELS 일치 확인 ✓  {actual_labels}')

PROJECT_LABELS 일치 확인 ✓  ['refrigerator', 'wash_tower', 'washer_dryer']


In [3]:
# ── Split 전 클래스별 이미지 수 확인 ───────────────────────────────────────────────
print('=== Split 전 현황 ===')
class_files = {}
for cls in sorted(os.listdir(config.PROCESSED_DIR)):
    cls_path = os.path.join(config.PROCESSED_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    files = sorted(
        os.path.join(cls_path, f)
        for f in os.listdir(cls_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    )
    if len(files) == 0:
        print(f'  {cls}: 0개 → 이미지 없음, 스킵')
        continue
    class_files[cls] = files
    print(f'  {cls}: {len(files)}개')

total_before = sum(len(v) for v in class_files.values())
print(f'\n합계: {total_before}개  |  클래스: {len(class_files)}개')

=== Split 전 현황 ===
  refrigerator: 251개
  wash_tower: 288개
  washer_dryer: 342개

합계: 881개  |  클래스: 3개


In [4]:
# ── Corrupt 이미지 검사 + phash 일괄 계산 (원본 수정 없음) ──────────────────────────
# corrupt/소형 이미지는 split 제외 목록에만 추가됩니다. 원본 파일은 건드리지 않습니다.

print('=== 이미지 검증 + phash 계산 ===')
hash_map    = {}  # {fpath: phash}
corrupt_log = []  # [(fpath, err)]
small_log   = []  # [(fpath, w, h)]

for cls, files in class_files.items():
    for fpath in tqdm(files, desc=cls, leave=False):
        try:
            img = Image.open(fpath).convert('RGB')
            w, h = img.size
            if w < config.MIN_CROP_SIZE or h < config.MIN_CROP_SIZE:
                small_log.append((fpath, w, h))
            else:
                hash_map[fpath] = imagehash.phash(img)
        except Exception as e:
            corrupt_log.append((fpath, str(e)))

print()
if corrupt_log:
    print(f'[경고] 열리지 않는 이미지 {len(corrupt_log)}개 (split 제외, 원본 보존):')
    for p, e in corrupt_log:
        print(f'  {os.path.basename(p)}: {e}')
else:
    print('Corrupt 이미지: 없음 ✓')

if small_log:
    print(f'\n[경고] 크기 미달 이미지 {len(small_log)}개 (< {config.MIN_CROP_SIZE}px, split 제외, 원본 보존):')
    for p, w, h in small_log:
        print(f'  {os.path.basename(p)}: {w}x{h}')
else:
    print('크기 미달 이미지: 없음 ✓')

exclude_paths = set(p for p, _ in corrupt_log) | set(p for p, _, _ in small_log)
print(f'\n유효 이미지: {len(hash_map)}개  |  제외: {len(exclude_paths)}개')

=== 이미지 검증 + phash 계산 ===


refrigerator:   0%|          | 0/251 [00:00<?, ?it/s]

wash_tower:   0%|          | 0/288 [00:00<?, ?it/s]

washer_dryer:   0%|          | 0/342 [00:00<?, ?it/s]


Corrupt 이미지: 없음 ✓
크기 미달 이미지: 없음 ✓

유효 이미지: 881개  |  제외: 0개


In [5]:
# ── 클래스 내 유사 중복 검사 (phash, 원본 유지 - 참고용) ─────────────────────────────
PHASH_THRESHOLD = 5  # 해밍 거리 ≤ 5 : 사실상 동일 이미지

print(f'=== 클래스 내 유사 중복 검사 (phash 거리 ≤ {PHASH_THRESHOLD}) ===')
dup_pairs = []

for cls, files in class_files.items():
    valid_files = [f for f in files if f in hash_map]
    for i in range(len(valid_files)):
        for j in range(i + 1, len(valid_files)):
            dist = hash_map[valid_files[i]] - hash_map[valid_files[j]]
            if dist <= PHASH_THRESHOLD:
                dup_pairs.append((cls, valid_files[i], valid_files[j], dist))

if dup_pairs:
    print(f'유사 중복 {len(dup_pairs)}쌍 발견 (원본 유지 — 직접 확인 후 삭제 여부 결정하세요):')
    for cls, p1, p2, dist in dup_pairs:
        print(f'  [{cls}] 거리={dist}')
        print(f'    {os.path.basename(p1)}')
        print(f'    {os.path.basename(p2)}')
else:
    print('유사 중복 없음 ✓')

=== 클래스 내 유사 중복 검사 (phash 거리 ≤ 5) ===


유사 중복 29쌍 발견 (원본 유지 — 직접 확인 후 삭제 여부 결정하세요):
  [refrigerator] 거리=4
    naver_0026.jpg
    naver_0048.jpg
  [refrigerator] 거리=2
    naver_0026.jpg
    naver_0065.jpg
  [refrigerator] 거리=4
    naver_0034.jpg
    naver_0098.jpg
  [refrigerator] 거리=2
    naver_0048.jpg
    naver_0065.jpg
  [refrigerator] 거리=2
    naver_0056.jpg
    naver_0140.jpg
  [refrigerator] 거리=4
    naver_0058.jpg
    naver_0109.jpg
  [refrigerator] 거리=4
    naver_0065.jpg
    naver_0098.jpg
  [washer_dryer] 거리=0
    naver_0000.jpg
    naver_0021.jpg
  [washer_dryer] 거리=2
    naver_0008.jpg
    naver_0034.jpg
  [washer_dryer] 거리=4
    naver_0012.jpg
    naver_0121.jpg
  [washer_dryer] 거리=2
    naver_0023.jpg
    naver_0114.jpg
  [washer_dryer] 거리=2
    naver_0023.jpg
    naver_0126.jpg
  [washer_dryer] 거리=4
    naver_0037.jpg
    naver_0084.jpg
  [washer_dryer] 거리=2
    naver_0037.jpg
    naver_0186.jpg
  [washer_dryer] 거리=4
    naver_0041.jpg
    naver_0089.jpg
  [washer_dryer] 거리=4
    naver_0041.jpg
    naver_0110.

In [6]:
# ── Stratified Split 8:1:1 ──────────────────────────────────────────────────────
all_paths, all_labels = [], []
for cls, files in class_files.items():
    for f in files:
        if f in hash_map:  # corrupt/소형 자동 제외
            all_paths.append(f)
            all_labels.append(cls)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)
valid_paths, test_paths, valid_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
)

split_result = {
    'train': list(zip(train_paths, train_labels)),
    'valid': list(zip(valid_paths, valid_labels)),
    'test':  list(zip(test_paths,  test_labels)),
}

from collections import Counter
print('=== Split 결과 ===')
for split_name, pairs in split_result.items():
    counts = Counter(lbl for _, lbl in pairs)
    print(f'  {split_name:5s}: {len(pairs):3d}개  {dict(counts)}')

=== Split 결과 ===
  train: 704개  {'refrigerator': 201, 'wash_tower': 230, 'washer_dryer': 273}
  valid:  88개  {'refrigerator': 25, 'wash_tower': 29, 'washer_dryer': 34}
  test :  89개  {'washer_dryer': 35, 'refrigerator': 25, 'wash_tower': 29}


In [7]:
# ── data/split/ 에 복사 (raw/processed 원본 절대 수정 안 함) ───────────────────────
if os.path.exists(SPLIT_DIR):
    print(f'[경고] {SPLIT_DIR} 이미 존재합니다. 삭제 후 재생성합니다 (seed 고정이므로 결과 동일).')
    shutil.rmtree(SPLIT_DIR)

for split_name, pairs in split_result.items():
    for src, cls in tqdm(pairs, desc=f'{split_name} 복사', leave=True):
        dst_dir = os.path.join(SPLIT_DIR, split_name, cls)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(src, os.path.join(dst_dir, os.path.basename(src)))

print('\n복사 완료 ✓')

[경고] data\split 이미 존재합니다. 삭제 후 재생성합니다 (seed 고정이므로 결과 동일).


train 복사:   0%|          | 0/704 [00:00<?, ?it/s]

valid 복사:   0%|          | 0/88 [00:00<?, ?it/s]

test 복사:   0%|          | 0/89 [00:00<?, ?it/s]


복사 완료 ✓


In [8]:
# ── Split 후 검증 ───────────────────────────────────────────────────────────────
print('=== Split 후 검증 ===\n')

# 1. 수량 일치
split_total = 0
print('── 클래스별 이미지 수 ──')
for split_name in ['train', 'valid', 'test']:
    print(f'  {split_name}:')
    for cls in sorted(class_files):
        p = os.path.join(SPLIT_DIR, split_name, cls)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        split_total += n
        print(f'    {cls}: {n}개')

print(f'\n  원본 유효 이미지: {len(all_paths)}개')
print(f'  split 합계      : {split_total}개')
match = len(all_paths) == split_total
print(f'  수량 일치: {"✓" if match else "[오류] 불일치!"}')

# 2. 파일명 교집합 (data leakage)
train_names = set(os.path.basename(p) for p, _ in split_result['train'])
valid_names = set(os.path.basename(p) for p, _ in split_result['valid'])
test_names  = set(os.path.basename(p) for p, _ in split_result['test'])

print('\n── 파일명 교집합 검사 ──')
for label, s in [('train∩valid', train_names & valid_names),
                 ('train∩test',  train_names & test_names),
                 ('valid∩test',  valid_names & test_names)]:
    status = '✓ 없음' if not s else f'[경고] {len(s)}개'
    print(f'  {label}: {status}')

# 3. phash cross-split 중복 검사
print(f'\n── Cross-split phash 검사 (거리 ≤ {PHASH_THRESHOLD}) ──')
split_hashes = {}
for split_name, pairs in split_result.items():
    split_hashes[split_name] = [(p, hash_map[p]) for p, _ in pairs if p in hash_map]

cross_leaks = []
for s1, s2 in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]:
    for p1, h1 in split_hashes[s1]:
        for p2, h2 in split_hashes[s2]:
            if h1 - h2 <= PHASH_THRESHOLD:
                cross_leaks.append((s1, os.path.basename(p1), s2, os.path.basename(p2), h1 - h2))

if cross_leaks:
    print(f'  [경고] Cross-split 유사 이미지 {len(cross_leaks)}쌍 (data leakage 위험):')
    for s1, f1, s2, f2, dist in cross_leaks:
        print(f'    {s1}↔{s2} 거리={dist}: {f1} / {f2}')
else:
    print('  Cross-split 유사 중복 없음 ✓')

print('\n=== Split 완료 ===')
print(f'결과: data/split/ 하위 train/valid/test 폴더')
print('다음 단계: 05_train_efficientnetv2.ipynb')

=== Split 후 검증 ===

── 클래스별 이미지 수 ──
  train:
    refrigerator: 201개
    wash_tower: 230개
    washer_dryer: 273개
  valid:
    refrigerator: 25개
    wash_tower: 29개
    washer_dryer: 34개
  test:
    refrigerator: 25개
    wash_tower: 29개
    washer_dryer: 35개

  원본 유효 이미지: 881개
  split 합계      : 881개
  수량 일치: ✓

── 파일명 교집합 검사 ──
  train∩valid: [경고] 26개
  train∩test: [경고] 39개
  valid∩test: [경고] 3개

── Cross-split phash 검사 (거리 ≤ 5) ──


  [경고] Cross-split 유사 이미지 11쌍 (data leakage 위험):
    train↔valid 거리=4: naver_0117.jpg / naver_0041.jpg
    train↔valid 거리=4: naver_0089.jpg / naver_0041.jpg
    train↔valid 거리=4: naver_0110.jpg / naver_0041.jpg
    train↔valid 거리=4: stg_0037_1.jpg / naver_0306.jpg
    train↔test 거리=2: naver_0140.jpg / naver_0056.jpg
    train↔test 거리=2: naver_0008.jpg / naver_0034.jpg
    train↔test 거리=4: naver_0098.jpg / naver_0034.jpg
    train↔test 거리=0: stg_0009.jpg / naver_0298.jpg
    train↔test 거리=4: naver_0102.jpg / naver_0063.jpg
    train↔test 거리=4: naver_0075.jpg / naver_0190.jpg
    valid↔test 거리=0: naver_0159.jpg / naver_0077.jpg

=== Split 완료 ===
결과: data/split/ 하위 train/valid/test 폴더
다음 단계: 05_train_efficientnetv2.ipynb
